# Vision API: Inductive and Deductive Coding of Images

This notebook demonstrates how to use the OpenAI Vision API to perform inductive and deductive coding on images, comparing visual features across coffee shop photographs.

In [ ]:
%pip install openai pandas nest_asyncio --upgrade --quiet

In [ ]:
import os
import json
import base64
import random
import asyncio
from getpass import getpass

import pandas as pd
import nest_asyncio
from openai import OpenAI, AsyncOpenAI

nest_asyncio.apply()

MODEL = "gpt-5.4-mini"

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()
async_client = AsyncOpenAI()

In [ ]:
# count the number of images in each folder in the photos folder
# path to the photos folder
path = "photos"

# list of folders in the photos folder
folders = os.listdir(path)[2:3] # limit for testing

# loop through the folders
total_images = 0
for folder in folders:
    # ignore if not a directory
    if not os.path.isdir(f"{path}/{folder}"):
        continue
    # get the length of the list of images in the folder
    num_images = len(os.listdir(f"{path}/{folder}"))
    print(f"{folder}: {num_images} images")
    total_images += num_images

print(f"Total images: {total_images}")

In [ ]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


def choose_image_pair():
    """Choose two random images from the photo folders."""
    all_images = []
    for folder in folders:
        if not os.path.isdir(f"photos/{folder}"):
            continue
        list_of_images = os.listdir(f"photos/{folder}")
        all_images.extend([f"photos/{folder}/{image}" for image in list_of_images])

    image_path_a = random.choice(all_images)
    image_path_b = random.choice(all_images)
    return image_path_a, image_path_b


image_path_a, image_path_b = choose_image_pair()

print(f"Image A: {image_path_a}")
print(f"Image B: {image_path_b}")
print()

# Getting the base64 string
base64_image_a = encode_image(image_path_a)
base64_image_b = encode_image(image_path_b)


def inductive_coding(base64_image_a, base64_image_b):
    """Use the Vision API to compare two coffee shop images via inductive coding."""
    user_prompt = """Compare these two coffee shops using inductive coding. What features, attributes, or elements of the design are similar or different? Give a JSONL response with a descriptive label name, coffee_shop_A and coffee_shop_B as keys, and one row per label. The value of the column for each coffee shop should be 1 if the label applies to this coffee shop, or zero if it doesn't. If the label applies to both coffee shops, it should be 1  in both columns. Only output the JSONL."""

    response = client.responses.create(
        model=MODEL,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": user_prompt},
                    {
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{base64_image_a}",
                    },
                    {
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{base64_image_b}",
                    },
                ],
            }
        ],
        max_output_tokens=500,
    )

    labels = response.output_text
    return labels


labels = inductive_coding(base64_image_a, base64_image_b)
print(labels)

In [ ]:
# create a list of dictionary with the filename, search keyword, label, value for each label
def clean_and_save_labels(labels):
    cleaned_labels = []
    for label in labels.split("\n"):
        label_json = json.loads(label)

        if label_json["coffee_shop_A"] == 1:
            cleaned_labels.append(
                {
                    "filename": image_path_a.split("/")[-1],
                    "folder": image_path_a.split("/")[-2],
                    "label": label_json["label"],
                }
            )
        if label_json["coffee_shop_B"] == 1:
            cleaned_labels.append(
                {
                    "filename": image_path_b.split("/")[-1],
                    "folder": image_path_b.split("/")[-2],
                    "label": label_json["label"],
                }
            )

    # if jsonl file exists, append to the file
    if os.path.exists("cleaned_labels.jsonl"):
        with open("cleaned_labels.jsonl", "a") as file:
            for label in cleaned_labels:
                file.write(json.dumps(label) + "\n")
    else:
        with open("cleaned_labels.jsonl", "w") as file:
            for label in cleaned_labels:
                file.write(json.dumps(label) + "\n")

    return cleaned_labels


cleaned_labels = clean_and_save_labels(labels)
print(cleaned_labels)

In [ ]:
async def process_label(image, folder, path, label):
    """Check if a single label applies to an image using the Vision API."""
    base64_image = encode_image(f"{path}/{folder}/{image}")
    user_prompt = f"""If the label \"{label['label']}\" applies to this image, return 1, otherwise return 0. Only output a 1 or 0."""

    response = await async_client.responses.create(
        model=MODEL,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": user_prompt},
                    {
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{base64_image}",
                    },
                ],
            }
        ],
        max_output_tokens=5,
    )

    score = response.output_text
    if score == "1":
        print(f"Label {label['label']}")
        return {
            "filename": image,
            "folder": folder,
            "label": label["label"],
        }
    else:
        return None


async def process_image(image, folder, path, cleaned_labels):
    """Process all labels for a single image."""
    print(f"Processing {image}")
    tasks = [process_label(image, folder, path, label) for label in cleaned_labels]
    result = await asyncio.gather(*tasks)
    print()
    return [label for label in result if label]


async def deductive_coding():
    """Apply labels from inductive coding across all images (deductive coding)."""
    with open("cleaned_labels.jsonl", "r") as file:
        cleaned_labels = [json.loads(line) for line in file]

    # remove duplicate labels
    cleaned_labels = list({label["label"]: label for label in cleaned_labels}.values())

    # limit to first 5 labels for testing
    cleaned_labels = cleaned_labels[0:5]

    processed_labels = []
    for folder in folders:
        if not os.path.isdir(f"{path}/{folder}"):
            continue
        for image in os.listdir(f"{path}/{folder}"):
            if not image.endswith(".jpeg"):
                continue
            result = await process_image(image, folder, path, cleaned_labels)
            processed_labels.extend(result)

    if os.path.exists("processed_labels.jsonl"):
        mode = "a"
    else:
        mode = "w"

    with open("processed_labels.jsonl", mode) as file:
        for label in processed_labels:
            file.write(json.dumps(label) + "\n")

    return processed_labels


async def main():
    processed_labels = await deductive_coding()
    print(f"There are {len(processed_labels)} labels. The first 5 labels are: {processed_labels[:5]}")


asyncio.run(main())

In [ ]:
df = pd.read_json("processed_labels.json")
df.head()

In [ ]:
# count for each label without 'Name: count, dtype: float64' at the bottom
label_counts = df["label"].value_counts() / total_images

for label, count in label_counts.items():
    print(f"{label}: {count}")

## Summary

- **Inductive coding**: We used the Vision API to compare pairs of coffee shop images and automatically generate descriptive labels (e.g., "Natural_light", "Wooden_tables") based on visual features.
- **Deductive coding**: We then applied the discovered labels across all images in the dataset using async API calls, scoring each image for the presence or absence of each label.
- **Analysis**: The resulting label frequencies reveal which design features are most common across coffee shops, enabling data-driven visual comparison at scale.